# Higher-resolution experiment — 1280 full-image inference

Stage-2 experiment of the project ([docs/proposal.md](../docs/proposal.md)):
**does full-image inference at `imgsz=1280` improve small-object detection over the
640 full-image baseline, and what does it cost?**

The two fine-tuned checkpoints (`yolo26m`, `yolo26x`) are reused **unchanged** —
training stays at `imgsz=640`; only the inference size changes. Each image is
letterboxed so that its **long side becomes 1280** (instead of 640) and passed to the
detector in one forward pass. Scoring uses the same single evaluator
[`src/evaluation/metrics.py`](../src/evaluation/metrics.py) with `imgsz=1280`, so every
metric matches the baseline and tiling protocol exactly.

Reported here:

1. what `imgsz=640` / `imgsz=1280` does to each native image size in the split
   (downscale vs **upscale** of images smaller than the inference size);
2. the **640 and 1280** full-image conditions for **both models**, with **all
   metrics** — 640 is recomputed here so that every condition is scored on the same
   checkpoints;
3. a quick 1280 − 640 delta (the full 640 / 1280 / tiling tables are built in
   [results.ipynb](results.ipynb));
4. *appendix* — every condition scored per native image size (supplementary table).

## 1. Setup

Fixed protocol constants (identical to the baseline and tiling notebooks) plus the
checkpoint paths. `EVAL_STAGE` (env var, default `finetune`) picks the checkpoints:
`finetune` = full 50-epoch runs (`runs/finetune/`), `baseline` = stage-1 5-epoch runs
(`runs/baseline/`). Outputs go to `runs/highres/<stage>/`.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from IPython.display import display
from ultralytics import YOLO

# ---- fixed protocol (identical to the baseline) --------------------------- #
SEED = 42
CONF_EVAL = 0.001  # AP confidence
IOU_NMS = 0.7  # NMS IoU, fixed for every condition

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import (
    IMG_SIZE,
    MANIFESTS_DIR,
    evaluate_split,
    load_split_paths,
)

# ---- high-resolution condition --------------------------------------------- #
SPLIT = "test"  # test-dev, 1,610 images (val is selection-only)
BASE_SIZE = IMG_SIZE  # 640: training size and baseline inference size
HIGH_SIZE = 1280  # higher-resolution full-image inference (proposal)
PREDICT_BATCH = 4  # accuracy pass only; 1280 x 16 can exceed 8 GB for yolo26x
RUNTIME_IMAGES = 200  # images used for the timed pass (same as tiling)
STRIDE = 32  # max YOLO stride: letterboxed sides are padded to a multiple of it

# ---- checkpoints: EVAL_STAGE=finetune (full runs) | baseline (stage-1) ------ #
RUNS_DIR = ROOT / "runs"
EVAL_STAGE = os.environ.get("EVAL_STAGE", "finetune")
WEIGHTS: dict[str, Path] = {
    name: RUNS_DIR / EVAL_STAGE / f"{name}_s{IMG_SIZE}" / "weights" / "best.pt"
    for name in ("yolo26m", "yolo26x")
}

HIGHRES_DIR = RUNS_DIR / "highres" / EVAL_STAGE
HIGHRES_DIR.mkdir(parents=True, exist_ok=True)
TILING_CSV = RUNS_DIR / "tiling" / EVAL_STAGE / f"tiling_scores_{SPLIT}_ov20.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"root={ROOT}")
print(f"stage={EVAL_STAGE} -> {HIGHRES_DIR}")
print(f"device={DEVICE}")
if DEVICE == "cuda":
    print(f"gpu={torch.cuda.get_device_name(0)}")
for name, path in WEIGHTS.items():
    print(f"{name}: {path} ({'found' if path.exists() else 'MISSING'})")

In [ ]:
def load_checkpoints(weights: dict[str, Path]) -> dict[str, YOLO]:
    """Load every checkpoint; fail early if a path is wrong."""
    models: dict[str, YOLO] = {}
    for name, path in weights.items():
        if not Path(path).exists():
            raise FileNotFoundError(f"{name}: checkpoint not found at {path}")
        models[name] = YOLO(str(path))
        print(f"{name}: loaded {len(models[name].names)} classes from {path}")
    return models


checkpoints = load_checkpoints(WEIGHTS)

## 2. What `imgsz` does to each native image size

`imgsz` is **not** a fixed network input: YOLO is fully convolutional and accepts any
size whose sides are multiples of the stride (32). At prediction time Ultralytics
letterboxes each image so that its **long side equals `imgsz`**, keeping the aspect
ratio, and pads the short side. Images whose long side is **smaller** than `imgsz`
are **upscaled** (Ultralytics' predictor letterbox uses `scaleup=True`): this
interpolates pixels but adds no new detail.

The table lists every native size in the split with its scale factor `r` at 640 and
1280 and the resulting network input (shown for a single image; inside a batch of
mixed sizes the short side is padded up to `imgsz`, which changes the padding only,
never the scale). `r > 1` means upscaling.

In [ ]:
def letterbox_shape(width: int, height: int, imgsz: int, stride: int = STRIDE) -> str:
    """Network input (w x h) for one image: long side -> imgsz, sides padded to stride."""
    r = imgsz / max(width, height)
    new_w, new_h = round(width * r), round(height * r)
    pad_w = -(-new_w // stride) * stride
    pad_h = -(-new_h // stride) * stride
    return f"{pad_w}x{pad_h}"


manifest = pd.read_csv(MANIFESTS_DIR / f"{SPLIT}.csv")
manifest["native"] = manifest["width"].astype(str) + "x" + manifest["height"].astype(str)
size_table = manifest.groupby(["width", "height", "native"]).size().rename("images").reset_index()
for imgsz in (BASE_SIZE, HIGH_SIZE):
    size_table[f"r@{imgsz}"] = imgsz / size_table[["width", "height"]].max(axis=1)
    size_table[f"input@{imgsz}"] = [
        letterbox_shape(w, h, imgsz) for w, h in zip(size_table["width"], size_table["height"])
    ]
size_table["upscaled@1280"] = size_table[f"r@{HIGH_SIZE}"] > 1
size_table = size_table.sort_values("images", ascending=False).reset_index(drop=True)
display(size_table.drop(columns=["width", "height"]).round(3))

upscaled = int(size_table.loc[size_table["upscaled@1280"], "images"].sum())
print(
    f"{upscaled}/{len(manifest)} {SPLIT} images ({upscaled / len(manifest):.1%}) are upscaled "
    f"at imgsz={HIGH_SIZE}; none are upscaled at imgsz={BASE_SIZE} "
    f"(min long side = {int(manifest[['width', 'height']].max(axis=1).min())} px)."
)

## 3. 640 and 1280 full-image evaluation

`evaluate_split(..., imgsz=640|1280)` letterboxes each full image to that long side,
runs one forward pass, maps boxes back to original-image coordinates, drops
predictions overlapping ignored regions and scores **all metrics** with the shared
evaluator. The runtime pass uses batch size 1, warm-up and CUDA synchronization on
the first `RUNTIME_IMAGES` images, exactly as in the tiling notebook.

In [ ]:
def run_full_eval(
    models: dict[str, YOLO],
    imgsz: int = HIGH_SIZE,
    split: str = SPLIT,
    runtime_images: int | None = RUNTIME_IMAGES,
) -> pd.DataFrame:
    """Full-image evaluation for every model at one inference size; writes one CSV."""
    images, labels = load_split_paths(split)
    rows: list[dict[str, Any]] = []
    for name, model in models.items():
        print(f"=== {name} full-image @ {imgsz} on {split} ({len(images)} images) ===")
        start = time.perf_counter()
        metrics = evaluate_split(
            labels=labels,
            images=images,
            model=model,
            imgsz=imgsz,
            conf=CONF_EVAL,
            batch=PREDICT_BATCH,
            runtime_images=runtime_images,
        )
        elapsed = time.perf_counter() - start
        rows.append(
            {
                "model": name,
                "weights": str(WEIGHTS[name]),
                "split": split,
                "inference": f"full-{imgsz}",
                "imgsz": imgsz,
                "runtime_images": len(images)
                if runtime_images is None
                else min(runtime_images, len(images)),
                **metrics,
            }
        )
        print(
            f"    mAP50-95={metrics['map50_95']:.4f} mAP50={metrics['map50']:.4f} "
            f"recall={metrics['recall50_25']:.4f} tiny_AP={metrics['ap_tiny']:.4f} "
            f"tiny_recall={metrics['recall_tiny']:.4f} "
            f"runtime={metrics['runtime_mean_ms']:.1f}ms ({elapsed:.0f}s total)"
        )
    table = pd.DataFrame(rows)
    out = HIGHRES_DIR / f"highres_scores_{split}_{imgsz}.csv"
    table.to_csv(out, index=False)
    print(f"saved -> {out}")
    return table


base_scores = run_full_eval(checkpoints, imgsz=BASE_SIZE, split=SPLIT)
highres_scores = run_full_eval(checkpoints, imgsz=HIGH_SIZE, split=SPLIT)
display(pd.concat([base_scores, highres_scores], ignore_index=True).T)

## 4. Quick look: 1280 − 640

Same checkpoints, same evaluator; positive is better for accuracy and worse for
runtime. The full result tables (overall, per disjoint size group, m vs x, with
tiling) are built in [results.ipynb](results.ipynb).

In [ ]:
quick_cols = [
    "map50_95",
    "map50",
    "recall50_25",
    "ap_tiny",
    "ap50_tiny",
    "recall_tiny",
    "ap_small_only",
    "recall_small_only",
    "ap_medium",
    "ap_large",
    "recall_large",
    "runtime_mean_ms",
    "runtime_p95_ms",
]
quick = highres_scores.set_index("model")[quick_cols] - base_scores.set_index("model")[quick_cols]
display(quick.round(4))

## 5. Summary

* The 640 and 1280 full-image conditions are evaluated for **both** models on the
  same checkpoints and written to `runs/highres/<stage>/`; every metric comes from the
  same shared `evaluate_split` used for tiling.
* `imgsz=1280` upscales every image whose long side is below 1280 (e.g. 960×540),
  so for those images it adds no new pixels; the appendix shows the per-size effect.
* 1280 vs tiling separates two causes of the tiling large-object regression: 1280
  enlarges objects (train/test scale mismatch) **without** cutting them at tile
  borders. If large AP drops at 1280 too, the scale mismatch contributes.
* Run order for a stage: this notebook → `tiling.ipynb` → `results.ipynb`
  (same `EVAL_STAGE` everywhere).
* All numbers follow our protocol and are **not** official VisDrone leaderboard scores.

## Appendix — per-native-size breakdown (supplementary)

**Not part of the main results.** The split-level numbers above average over images
of different native sizes, and the conditions treat them differently (section 2):
`imgsz=1280` upscales images with a long side below 1280, and tiling needs 2 crops
for a 960×540 image but 8 for a 1920×1080 one. Here every condition (640 / 1280 /
tiling 20 %) is scored **separately on each native-size group** with all shared
metrics plus runtime, via the standalone
[`src/evaluation/native_size.py`](../src/evaluation/native_size.py) (it reuses the
shared `score_predictions`; the main evaluator is unchanged).

Groups differ in camera, altitude and scene, so compare conditions **within** a
group (delta table), not groups against each other. This costs one extra prediction
pass per model and condition.

In [ ]:
from src.evaluation.native_size import (
    SIZE_METRICS,
    delta_vs_reference,
    evaluate_by_native_size,
)

# Optional extra pass (~30-60 min for both models): RUN_APPENDIX=0 skips it.
RUN_APPENDIX = os.environ.get("RUN_APPENDIX", "1") == "1"
count_cols = ["n_all", "n_tiny", "n_small_only", "n_medium", "n_large"]

if RUN_APPENDIX:
    images, labels = load_split_paths(SPLIT)
    by_size = evaluate_by_native_size(checkpoints, images, labels, batch=PREDICT_BATCH)
    by_size.to_csv(HIGHRES_DIR / f"appendix_by_native_size_{SPLIT}.csv", index=False)

    print("Images and objects per native-size group:")
    first_model = next(iter(WEIGHTS))
    display(
        by_size[(by_size["inference"] == "full-640") & (by_size["model"] == first_model)][
            ["native", "images", *count_cols]
        ].astype(dict.fromkeys(count_cols, int))
    )
    for model in WEIGHTS:
        print(f"{model}: all metrics per native size and condition")
        display(
            by_size[by_size["model"] == model]
            .set_index(["native", "inference"])[list(SIZE_METRICS)]
            .round(4)
        )

    size_deltas = delta_vs_reference(by_size, reference="full-640")
    size_deltas.to_csv(HIGHRES_DIR / f"appendix_by_native_size_delta_{SPLIT}.csv")
    print("Delta vs 640 inside each native-size group:")
    display(size_deltas.round(4))
else:
    print("appendix skipped (RUN_APPENDIX=0)")